# Implementasi Bab 7 — Evaluasi Model Klasifikasi

Confusion matrix, precision, recall, F1, ROC-AUC, PR curve, dan Stratified k-Fold CV pada data tidak seimbang [ML-MODUL-01, hlm. 53–54].


In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    auc,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
X, y = make_classification(
    n_samples=1000, n_features=10, weights=[0.7, 0.3], random_state=SEED
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=SEED, stratify=y
)
model = LogisticRegression(max_iter=2000, random_state=SEED).fit(X_train, y_train)
y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]


In [ ]:
matrix = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", matrix)
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall:", round(recall_score(y_test, y_pred), 3))
print("F1:", round(f1_score(y_test, y_pred), 3))
print(classification_report(y_test, y_pred))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_score)
precision, recall, _ = precision_recall_curve(y_test, y_score)
roc_auc = auc(fpr, tpr)
pr_auc = auc(recall, precision)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(fpr, tpr, label=f"AUC={roc_auc:.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set(title="ROC Curve", xlabel="FPR", ylabel="TPR")
axes[0].legend()
axes[1].plot(recall, precision, label=f"PR-AUC={pr_auc:.3f}")
axes[1].set(title="Precision-Recall Curve", xlabel="Recall", ylabel="Precision")
axes[1].legend()
plt.show()


In [ ]:
stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_scores = cross_val_score(model, X, y, cv=stratified_cv, scoring="f1", n_jobs=-1)
print("F1 per fold:", np.round(cv_scores, 3))
print(f"Mean={cv_scores.mean():.3f}, std={cv_scores.std():.3f}")


## Latihan 1: laporan dua model pada data medis

Dataset Breast Cancer dipakai sebagai latihan akademik, bukan alat diagnosis. Logistic Regression dibungkus scaling dalam Pipeline agar CV bebas leakage [ML-MODUL-01, hlm. 54].


In [ ]:
cancer = load_breast_cancer()
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    cancer.data, cancer.target, test_size=0.2, random_state=SEED, stratify=cancer.target
)
report_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=SEED)),
    ]),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=SEED),
}
rows = []
exercise_fig, exercise_axes = plt.subplots(2, 3, figsize=(15, 8))
for row_index, (name, candidate) in enumerate(report_models.items()):
    candidate.fit(X_train_c, y_train_c)
    prediction = candidate.predict(X_test_c)
    probability = candidate.predict_proba(X_test_c)[:, 1]
    rows.append((
        name,
        precision_score(y_test_c, prediction),
        recall_score(y_test_c, prediction),
        f1_score(y_test_c, prediction),
    ))
    ConfusionMatrixDisplay.from_predictions(
        y_test_c, prediction, cmap="Blues", ax=exercise_axes[row_index, 0]
    )
    exercise_axes[row_index, 0].set_title(f"Confusion Matrix — {name}")
    exercise_roc = roc_curve(y_test_c, probability)
    exercise_axes[row_index, 1].plot(exercise_roc[0], exercise_roc[1])
    exercise_axes[row_index, 1].plot([0, 1], [0, 1], "--", color="gray")
    exercise_axes[row_index, 1].set_title(f"ROC — {name}")
    exercise_pr = precision_recall_curve(y_test_c, probability)
    exercise_axes[row_index, 2].plot(exercise_pr[1], exercise_pr[0])
    exercise_axes[row_index, 2].set_title(f"Precision-Recall — {name}")
plt.tight_layout()
plt.show()
display(pd.DataFrame(rows, columns=["model", "precision", "recall", "f1"]))


In [ ]:
model_list = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=SEED)),
    ]),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=SEED),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5)),
    ]),
}
comparison = []
for name, candidate in model_list.items():
    scores = cross_val_score(candidate, X, y, cv=stratified_cv, scoring="f1", n_jobs=-1)
    comparison.append((name, scores.mean(), scores.std()))
display(pd.DataFrame(comparison, columns=["model", "mean_f1", "std_f1"]).sort_values("mean_f1", ascending=False))


## Kesimpulan

ROC-AUC mengukur ranking lintas threshold, sedangkan PR curve lebih informatif saat kelas positif langka. Mean dan standar deviasi CV perlu dibaca bersama untuk menilai performa sekaligus stabilitas [ML-MODUL-01, hlm. 54].
